# Zadanie z ćwiczeń 12 i 13

Proszę przeprowadzić analizę danych z wykorzystaniem narzędzi poznanych na zajęciach, wliczając jedną z bibliotek: Pandas lub Polars.

Proszę uzupełnić brakujące miejsca w skrypcie i przesłać go przez system Moodle.


## Opis danych


W ramach tego ćwiczenia przeanalizujemy rzeczywiste dane uzyskane przy użyciu zaawansowanej technologii **NGS (next-generation sequencing)**. Dane te dotyczą drożdży z gatunku S. pombe, które są często wykorzystywane jako model badawczy ze względu na swoją prostotę i dobrze poznany genom. Pozwala to na prowadzenie różnorodnych eksperymentów i analiz na poziomie molekularnym.

(Fot Jürgen Berger/Max Planck Institute for Developmental Biology)


![pombe.jpg](data/figures/pombe.jpg)

## Przygotowanie danych
Dane przeszły już wstępną analizę zgodnie z protokołem metody ["genome-wide 3'RACE"](https://www.nature.com/articles/s41467-024-50824-w). W tabeli zostały zaczerpnięte informacje na temat mapowania obydwu nici do genomu referencyjnego, natomiast zastosowana metoda miała na celu identyfikację i analizę końców 3' mRNA - z uwzględnieniem występowania ogonów poly(A), a także modyfikacji tych ogonów poprzez dodatnie terminalnych (końcowych) reszt urydyn.

Załączone dane stanowią mały wycinek eksperymentu, w którym spróbujemy policzyć długość ogonów poly(A) i urydylowanych ogonów poly(A): poly(AU) oraz oligo(U) - ktore zawierają same urydyny na końcu.


In [ ]:
!python3 download_data.py


## Import bibliotek

In [ ]:
# Proszę zaimportować niezbędne biblioteki do analizy i wizualizacji danych (0,5 pkt)

import polars as pl
import plotly.express as px


In [ ]:
## Proszę wczytać plik csv "data_RNAtails_wt.csv" (1 pkt)

df_RNAtails = pl.read_csv("data/data_RNAtails_wt.csv", separator=",")


In [ ]:
## Zad 1: Proszę sprawdzić jaką strukturę mają dane - proszę podać ile występuje kolumn i wierszy w tabeli (1pkt)

print(f"Liczba wierszy: {df_RNAtails.height}")
print(f"Liczba kolumn: {df_RNAtails.width}")


In [ ]:
df_RNAtails.columns

In [ ]:
## Zad 2: Proszę sprawdzić jaki jest rozkład danych w kolumnie "distance_to_TES".


# W tej kolumnie znajdują się informacje w jakiej odległości od miejsca cięcia i poliadenylacji (TES) znajdują się zmapowane końce 3' mRNA.
# W dużym uproszczeniu końce 3' powinny zostać zmapowane dokładnie  w TES (w odległości 0 od TES).


## 2A) Proszę wypisać wartości z kolumny "distance_to_TES", które występują najczęściej, wraz z liczbą ich występowania w tabeli (count). (1pkt)

df_RNAtails.group_by("distance_to_TES").agg([pl.col("distance_to_TES").count().alias("count")]).sort("count", descending=True)


In [ ]:
## 2B) Proszę przefiltrować tabelę - pozostawić tylko te odczyty, które w kolumnie "distance_to_TES" mają wartości od -3 do 3. (2pkt)

df_RNAtails_filtrated = df_RNAtails.filter((pl.col("distance_to_TES") >= -3), (pl.col("distance_to_TES") <= 3))

df_RNAtails_filtrated


In [ ]:
## 2C) Proszę podać liczbę rzędów po filtracji oraz policzyć ile % odczytów odfiltrowano (1 pkt)

rows = df_RNAtails.height
rows_filtrated = df_RNAtails_filtrated.height

print(f"Liczba wierszy przed filtracją: {rows}")
print(f"Liczba wierszy po filtracji: {rows_filtrated}")
print(f"Odfiltrowano: {round(100-rows_filtrated*100/rows, 2)} % odczytów")


In [ ]:
## Zad 3:
## 3A) W odfiltrowanej tabeli proszę sprawdzić liczbę wystąpień unikalnych wartości w kolumnie 'tail_type' (1 pkt)

df_RNAtails_filtrated['tail_type'].value_counts()


In [ ]:
## 3B) Proszę pozostawić tylko rzędy, które w kolumnie 'tail_type' zawierają warości: 'polyA', 'polyAU' lub 'oligoU' (1,5 pkt)

df_RNAtails_filtrated = df_RNAtails_filtrated.filter(pl.col("tail_type").is_in(["polyA", "polyAU",  "oligoU"]))
df_RNAtails_filtrated


In [ ]:
## Zad 4: Proszę dodać nową kolumnę 'tail_len', w której policzysz liczbę znaków w kolumnie 'tail' (2 pkt)
# w celu przeprowadzenia tego na całej tabeli (wszystkich wierszach), można skorzystać np z funkcji apply i lambda - proszę poszukać w dokumentacji.

df_RNAtails_filtrated = df_RNAtails_filtrated.with_columns([
    pl.col('tail').map_elements(lambda x: len(x), return_dtype=pl.Int32).alias('tail_len')
])

df_RNAtails_filtrated['tail', 'tail_len']


In [ ]:
## Zad 5: Dla każdego z typów ogonów 'polyA', 'polyAU' lub 'oligoU':

## 5A) Policz wartość minimalną, maksymalną i medianę (1 pkt) (czego ? - Max: zakłądam 'tail_len' ponieważ po filtarcji nie ma innej sensownej kolumny)
chosen_cols = ['tail_len']
df_RNAtails_filtrated.group_by('tail_type').agg(
    pl.min(chosen_cols).name.prefix('min_'),
    pl.max(chosen_cols).name.prefix('max_'),
    pl.median(chosen_cols).name.prefix('median_')
)


In [ ]:
## 5B. Narysuj histogram rozkładu długości ogonów 'polyA', 'polyAU' lub 'oligoU' - proszę to przedstawić na jednym wykresie (2 pkt)

# sanitation check - if earlier cell is missed
df_plot = df_RNAtails_filtrated.filter(
    pl.col('tail_type').is_in(['polyA', 'polyAU', 'oligoU'])
)

fig = px.histogram(
    df_plot,
    x='tail_len',
    color='tail_type',
    barmode="overlay",
    histnorm='probability density',
    category_orders={"tail_type": ["polyA", "polyAU", "oligoU"]}
)
fig.update_layout(
    yaxis_title='Estymowana gęstość prawdopodobieństwa',
    xaxis_title='Długość ogona',
    legend_title='Rodzaj ogona'
)
fig.show()
# df_RNAtails_filtrated.schema